# backtesting.py Orchestrator Tutorial

This notebook shows how to use `quant-orchestrator` with `quant-warehouse` for multi-vendor, single-symbol and multi-symbol research, Monte Carlo simulation, walk-forward optimization, equity-curve analysis, and portfolio optimization.


In [1]:

from __future__ import annotations

from pathlib import Path
import sys
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.experiments import WindowSpec
from quant_orchestrator.monte_carlo import simulate_return_paths
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.data_adapter import build_backtesting_frame
from quant_orchestrator.platforms.backtesting_frameworks.shared import MAG7_SYMBOLS, load_price_frame, load_signal_frame, normalize_session_label
from quant_orchestrator.strategy import summarize_backtest, summarize_equity
from quant_warehouse.feature_engineering import compute_features_worldclass

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

SINGLE_SYMBOL = 'AAPL'
MULTI_VENDOR_PROVIDERS = ('fmp', 'yfinance')
TRAIN_START = '2020-01-01'
TRAIN_END = '2025-12-31'
TEST_START = '2026-01-01'
TEST_END = None
FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    if fast_window >= slow_window:
        raise ValueError('fast_window must be less than slow_window')

    frame = compute_features_worldclass(prices.copy())
    fast_col = f'SMA{fast_window}'
    slow_col = f'SMA{slow_window}'
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(
            'Quant Warehouse feature output is missing required SMA columns: '
            f'{missing}. Update quant-warehouse feature engineering first.'
        )

    frame['fast_sma'] = frame[fast_col]
    frame['slow_sma'] = frame[slow_col]
    frame['signal'] = (frame['fast_sma'] > frame['slow_sma']).astype(int).fillna(0)
    frame = frame.dropna(subset=['open', 'high', 'low', 'close', 'volume']).copy()
    return frame



def combine_equity_sleeves(sleeves: list[pd.Series]) -> pd.Series:
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index, name='portfolio_value')
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index).ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index()


def align_return_frame(equity_map: dict[str, pd.Series]) -> pd.DataFrame:
    return pd.concat(
        {name: equity.pct_change().fillna(0.0) for name, equity in equity_map.items()},
        axis=1,
    ).fillna(0.0)


def max_sharpe_weights(return_frame: pd.DataFrame) -> pd.Series:
    if return_frame.empty:
        raise ValueError('return_frame cannot be empty')
    columns = list(return_frame.columns)
    mean = return_frame.mean().to_numpy(dtype=float)
    cov = return_frame.cov().to_numpy(dtype=float)
    if len(columns) == 1:
        return pd.Series([1.0], index=columns)

    def neg_sharpe(weights: np.ndarray) -> float:
        portfolio_mean = float(weights @ mean)
        portfolio_vol = float(np.sqrt(weights @ cov @ weights))
        if portfolio_vol <= 0:
            return 1e9
        return -(portfolio_mean / portfolio_vol)

    bounds = [(0.0, 1.0)] * len(columns)
    constraints = ({'type': 'eq', 'fun': lambda weights: float(np.sum(weights)) - 1.0},)
    guess = np.repeat(1.0 / len(columns), len(columns))
    result = minimize(neg_sharpe, guess, bounds=bounds, constraints=constraints, method='SLSQP')
    weights = pd.Series(result.x if result.success else guess, index=columns)
    weights = weights.clip(lower=0.0)
    return weights / weights.sum()


def weighted_equity_from_returns(return_frame: pd.DataFrame, weights: pd.Series, *, capital_base: float) -> pd.Series:
    aligned = return_frame.loc[:, weights.index].fillna(0.0)
    portfolio_returns = aligned.mul(weights, axis=1).sum(axis=1)
    equity = capital_base * (1.0 + portfolio_returns).cumprod()
    return equity.rename('portfolio_value')


def wfo_windows() -> WindowSpec:
    return WindowSpec(
        mode='fixed',
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        test_start=TEST_START,
        test_end=TEST_END,
    )


def run_monte_carlo(equity: pd.Series, *, iterations: int = 1000, block_size: int = 5) -> pd.DataFrame:
    returns = equity.pct_change().dropna()
    mc = simulate_return_paths(returns, iterations=iterations, block_size=block_size, horizon=len(returns))
    return mc.summary


def summarize_candidates(rows: list[dict[str, object]]) -> pd.DataFrame:
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    return table.sort_values(by=['total_return', 'max_drawdown', 'final_equity'], ascending=[False, True, False]).reset_index(drop=True)

FRAMEWORK_TITLE = 'backtesting.py'


def run_framework_backtest(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from backtesting import Backtest, Strategy

    trade_size = max(1, int((capital_base * 0.25) / float(frame['close'].iloc[0])))
    bt_frame = build_backtesting_frame(frame).drop(columns=['Signal'])
    signal_map = {normalize_session_label(date): bool(signal) for date, signal in frame['signal'].items()}

    class SignalStrategy(Strategy):
        def init(self):
            self.trade_size = trade_size

        def next(self):
            bullish = signal_map.get(normalize_session_label(self.data.index[-1]), False)
            if bullish and not self.position:
                self.buy(size=self.trade_size)
            elif not bullish and self.position:
                self.position.close()

    started = perf_counter()
    stats = Backtest(
        bt_frame,
        SignalStrategy,
        cash=capital_base,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
    ).run()
    elapsed = perf_counter() - started
    equity = stats['_equity_curve']['Equity'].rename('portfolio_value')
    summary = summarize_backtest(
        framework='backtesting.py',
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(bt_frame),
        trades=int(stats['# Trades']),
    )
    summary['native_return_pct'] = float(stats['Return [%]'])
    summary['native_sharpe'] = float(stats['Sharpe Ratio']) if pd.notna(stats['Sharpe Ratio']) else None
    summary['native_max_drawdown_pct'] = float(stats['Max. Drawdown [%]'])
    return stats, summary, equity


## Single Symbol / Multi Vendor


In [2]:

print(f'Tutorial framework: {FRAMEWORK_TITLE}')
print(f'Single symbol: {SINGLE_SYMBOL}')
print(f'Vendors: {MULTI_VENDOR_PROVIDERS}')


Tutorial framework: backtesting.py
Single symbol: AAPL
Vendors: ('fmp', 'yfinance')


## Multi Vendor Backtest


In [3]:

def run_multi_vendor_single_symbol(symbol: str = SINGLE_SYMBOL, providers: tuple[str, ...] = MULTI_VENDOR_PROVIDERS, capital_base: float = 100_000.0) -> pd.DataFrame:
    rows = []
    for provider in providers:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        rows.append(summary.assign(provider=provider))
    return pd.concat(rows, ignore_index=True)

vendor_report = run_multi_vendor_single_symbol()
print(vendor_report.to_string(index=False))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

     framework symbol      start        end  bars  trades  final_equity  total_return  max_drawdown  daily_vol  elapsed_seconds  bars_per_second  native_return_pct  native_sharpe  native_max_drawdown_pct provider
backtesting.py   AAPL 2020-10-15 2026-06-24  1428       5     113054.77        0.1305       -0.1414     0.0052           0.0253         56459.90          13.054770       0.260723                -14.14134      fmp
backtesting.py   AAPL 2020-10-15 2026-06-23  1427       5     113753.08        0.1375       -0.1381     0.0050           0.0231         61896.15          13.753082       0.280719                -13.81320 yfinance


## Multi Symbol Backtest


In [4]:

def run_multi_symbol_backtest(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    symbol_rows = []
    sleeves = []
    runs = {}
    elapsed = 0.0
    trades = 0
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        started = perf_counter()
        raw, summary, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        elapsed += perf_counter() - started
        symbol_rows.append(summary.assign(provider=provider))
        sleeves.append(equity.rename(symbol))
        trades += int(summary['trades'].iloc[0])
        runs[symbol] = raw
    portfolio_equity = combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol='MAG7',
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary['provider'] = provider
    return pd.concat(symbol_rows, ignore_index=True), portfolio_summary, runs, portfolio_equity

symbol_report, portfolio_report, runs_by_symbol, portfolio_equity = run_multi_symbol_backtest()
print('Symbol report:')
print(symbol_report.loc[:, ['provider', 'symbol', 'start', 'end', 'bars', 'trades', 'final_equity', 'total_return', 'max_drawdown', 'elapsed_seconds']].to_string(index=False))
print()
print('Portfolio report:')
print(portfolio_report.to_string(index=False))


Symbol report:
provider symbol      start        end  bars  trades  final_equity  total_return  max_drawdown  elapsed_seconds
     fmp   AAPL 2020-10-15 2026-06-24  1428       5      16124.41        0.1287       -0.1397           0.0231
     fmp   AMZN 2020-10-15 2026-06-24  1428       5      13895.53       -0.0273       -0.1444           0.0227
     fmp  GOOGL 2020-10-15 2026-06-24  1428       3      25466.41        0.7826       -0.1377           0.0250
     fmp   META 2020-10-15 2026-06-24  1428       3      19721.66        0.3805       -0.1505           0.0233
     fmp   MSFT 2020-10-15 2026-06-24  1428       5      16334.03        0.1434       -0.1197           0.0233
     fmp   NVDA 2020-10-15 2026-06-24  1428       3      51913.51        2.6339       -0.2184           0.0231
     fmp   TSLA 2020-10-15 2026-06-24  1428       6      13070.39       -0.0851       -0.3521           0.0223

Portfolio report:
     framework symbol      start        end  bars  trades  final_equity  total

## Monte Carlo


In [5]:

mc_report = run_monte_carlo(portfolio_equity, iterations=1000, block_size=5)
print(mc_report.to_string(index=False))


 iterations  horizon  terminal_return_mean  terminal_return_p05  terminal_return_p50  terminal_return_p95  max_drawdown_mean  max_drawdown_p05
       1000     1427              0.596455             0.085814             0.550308             1.238193          -0.149366         -0.240323


## Walk Forward Optimization


In [6]:

def run_walk_forward_optimization(symbol: str = SINGLE_SYMBOL, provider: str = 'fmp', capital_base: float = 100_000.0):
    spec = wfo_windows()
    train_frame = load_price_frame(symbol, provider=provider, start=spec.train_start, end=spec.train_end)
    test_frame = load_price_frame(symbol, provider=provider, start=spec.test_start, end=spec.test_end)

    train_rows = []
    for fast_window, slow_window in product(FAST_WINDOWS, SLOW_WINDOWS):
        if fast_window >= slow_window:
            continue
        frame = build_sma_frame(train_frame, fast_window=fast_window, slow_window=slow_window)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        row = summary.iloc[0].to_dict()
        row['fast_window'] = fast_window
        row['slow_window'] = slow_window
        train_rows.append(row)

    train_grid = summarize_candidates(train_rows)
    best = train_grid.iloc[0]
    best_fast = int(best['fast_window'])
    best_slow = int(best['slow_window'])
    _, _, test_equity = run_framework_backtest(build_sma_frame(test_frame, fast_window=best_fast, slow_window=best_slow), symbol=symbol, capital_base=capital_base)
    test_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol=symbol,
        equity=test_equity,
        elapsed_seconds=0.0,
        bars=len(test_frame),
        trades=None,
    )
    return train_grid, best, test_summary

train_grid, best_row, test_summary = run_walk_forward_optimization()
print('Train grid:')
print(train_grid.loc[:, ['fast_window', 'slow_window', 'final_equity', 'total_return', 'max_drawdown']].to_string(index=False))
print()
print('Best parameters:')
print(best_row.loc[['fast_window', 'slow_window', 'total_return', 'max_drawdown']].to_string())
print()
print('Test summary:')
print(test_summary.to_string(index=False))


Train grid:
 fast_window  slow_window  final_equity  total_return  max_drawdown
          50          100     145219.15        0.4522       -0.1230
          10          200     141427.60        0.4143       -0.1291
          10          100     140844.55        0.4084       -0.1643
          20          100     139499.05        0.3950       -0.1823
          20          200     133051.00        0.3305       -0.1887
          10          150     127920.85        0.2792       -0.1659
          50          150     126302.80        0.2630       -0.1472
          20          150     125202.25        0.2520       -0.1480
          50          200     112834.00        0.1283       -0.2120

Best parameters:
fast_window         50
slow_window        100
total_return    0.4522
max_drawdown    -0.123

Test summary:
     framework symbol      start        end  bars trades  final_equity  total_return  max_drawdown  daily_vol  elapsed_seconds bars_per_second
backtesting.py   AAPL 2026-01-02 2026-06

## Equity Curve Analysis


In [7]:

print('Equity curve summary:')
print(summarize_equity(portfolio_equity).items())
print()
print(portfolio_equity.tail().to_string())


Equity curve summary:
dict_items([('start', '2020-10-15'), ('end', '2026-06-24'), ('final_equity', 156525.97), ('total_return', 0.5653), ('max_drawdown', -0.1255), ('daily_vol', 0.0066)])

2026-06-17 00:00:00    158439.22
2026-06-18 00:00:00    160376.71
2026-06-22 00:00:00    158757.16
2026-06-23 00:00:00    156348.28
2026-06-24 00:00:00    156525.97


## Portfolio Optimization


In [8]:

def optimize_symbol_portfolio(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    equity_map = {}
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, _, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        equity_map[symbol] = equity
    return equity_map

symbol_equities = optimize_symbol_portfolio()
symbol_returns = align_return_frame(symbol_equities)
symbol_weights = max_sharpe_weights(symbol_returns)
symbol_portfolio_equity = weighted_equity_from_returns(symbol_returns, symbol_weights, capital_base=100_000.0)
print('Symbol-level weights:')
print(symbol_weights.sort_values(ascending=False).to_string())
print()
print('Symbol-level portfolio summary:')
print(summarize_equity(symbol_portfolio_equity))

candidate_pairs = [(fast, slow) for fast in FAST_WINDOWS for slow in SLOW_WINDOWS if fast < slow]
strategy_equities = {}
strategy_rows = []
strategy_frame = load_price_frame(SINGLE_SYMBOL, provider='fmp', start=TRAIN_START, end=TEST_END)
for fast_window, slow_window in candidate_pairs:
    candidate_frame = build_sma_frame(strategy_frame, fast_window=fast_window, slow_window=slow_window)
    _, summary, equity = run_framework_backtest(candidate_frame, symbol=SINGLE_SYMBOL, capital_base=100_000.0)
    key = f'{fast_window}/{slow_window}'
    strategy_equities[key] = equity
    strategy_rows.append({
        'strategy': key,
        'fast_window': fast_window,
        'slow_window': slow_window,
        'total_return': float(summary['total_return'].iloc[0]),
        'max_drawdown': float(summary['max_drawdown'].iloc[0]),
    })
strategy_returns = align_return_frame(strategy_equities)
strategy_weights = max_sharpe_weights(strategy_returns)
strategy_portfolio_equity = weighted_equity_from_returns(strategy_returns, strategy_weights, capital_base=100_000.0)
print()
print('Strategy-level weights:')
print(strategy_weights.sort_values(ascending=False).head(10).to_string())
print()
print('Strategy-level portfolio summary:')
print(summarize_equity(strategy_portfolio_equity))


Symbol-level weights:
GOOGL    5.628444e-01
META     2.247941e-01
NVDA     2.123615e-01
TSLA     5.165471e-17
MSFT     3.435326e-17
AAPL     0.000000e+00
AMZN     0.000000e+00

Symbol-level portfolio summary:
{'start': '2020-10-15', 'end': '2026-06-24', 'final_equity': 202487.45, 'total_return': 1.0249, 'max_drawdown': -0.1162, 'daily_vol': 0.0064}



Strategy-level weights:
10/200    5.656120e-01
50/100    4.343880e-01
10/100    2.081668e-16
10/150    0.000000e+00
20/100    0.000000e+00
20/150    0.000000e+00
20/200    0.000000e+00
50/150    0.000000e+00
50/200    0.000000e+00

Strategy-level portfolio summary:
{'start': '2020-01-02', 'end': '2026-06-24', 'final_equity': 146900.23, 'total_return': 0.469, 'max_drawdown': -0.1177, 'daily_vol': 0.006}
